# 01 · nnU-Net preprocessing — 600-case subset (BraTS2023 GLI)

Full preprocessing of all ~1061 training cases produced ~23 GB, over the 20 GB
limit of `/kaggle/working`, so the run failed partway. This version preprocesses
a fixed **600-case subset** (500 train + 100 val, seed 42, patient-separated)
instead, which lands at ~13 GB — comfortably under the limit.

The mounted dataset has its `.nii.gz` auto-decompressed by Kaggle into `.nii`,
with the **GLI training images nested** inside folders (`X_0000.nii/inner.nii`)
and the labels flat (`X.nii`). We rebuild a clean nnU-Net raw tree of **symlinks**
(no copying) for only the 600 subset cases, validate it, then run
`nnUNetv2_plan_and_preprocess`.

In [ ]:
# Step 1 — environment: disk, free space, cpu count
import os, subprocess, shutil
print("=== df -h ===")
print(subprocess.run(["df","-h"], capture_output=True, text=True).stdout)
for p in ["/kaggle/working", "/tmp"]:
    try:
        u = shutil.disk_usage(p)
        print(f"{p:16s} free {u.free/1e9:6.1f} GB / total {u.total/1e9:6.1f} GB")
    except Exception as e:
        print(f"{p:16s} n/a ({e})")
print("os.cpu_count():", os.cpu_count())
print("nproc:", subprocess.run(["nproc"], capture_output=True, text=True).stdout.strip())

In [ ]:
# Step 2 — install nnunetv2
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "nnunetv2"], check=True)
from importlib.metadata import version
print("nnunetv2 version:", version("nnunetv2"))

In [ ]:
TRAIN = [
    "BraTS-GLI-00000-000",
    "BraTS-GLI-00002-000",
    "BraTS-GLI-00006-000",
    "BraTS-GLI-00008-000",
    "BraTS-GLI-00008-001",
    "BraTS-GLI-00012-000",
    "BraTS-GLI-00014-000",
    "BraTS-GLI-00017-000",
    "BraTS-GLI-00019-000",
    "BraTS-GLI-00020-000",
    "BraTS-GLI-00021-000",
    "BraTS-GLI-00021-001",
    "BraTS-GLI-00024-000",
    "BraTS-GLI-00024-001",
    "BraTS-GLI-00025-000",
    "BraTS-GLI-00026-000",
    "BraTS-GLI-00028-000",
    "BraTS-GLI-00031-001",
    "BraTS-GLI-00033-000",
    "BraTS-GLI-00036-000",
    "BraTS-GLI-00044-000",
    "BraTS-GLI-00051-000",
    "BraTS-GLI-00053-001",
    "BraTS-GLI-00056-001",
    "BraTS-GLI-00058-001",
    "BraTS-GLI-00062-000",
    "BraTS-GLI-00066-000",
    "BraTS-GLI-00068-000",
    "BraTS-GLI-00071-000",
    "BraTS-GLI-00081-000",
    "BraTS-GLI-00088-000",
    "BraTS-GLI-00088-001",
    "BraTS-GLI-00095-000",
    "BraTS-GLI-00095-001",
    "BraTS-GLI-00096-001",
    "BraTS-GLI-00098-000",
    "BraTS-GLI-00098-001",
    "BraTS-GLI-00099-000",
    "BraTS-GLI-00102-000",
    "BraTS-GLI-00107-000",
    "BraTS-GLI-00108-000",
    "BraTS-GLI-00109-000",
    "BraTS-GLI-00111-000",
    "BraTS-GLI-00113-000",
    "BraTS-GLI-00115-000",
    "BraTS-GLI-00116-000",
    "BraTS-GLI-00117-000",
    "BraTS-GLI-00120-000",
    "BraTS-GLI-00121-000",
    "BraTS-GLI-00123-000",
    "BraTS-GLI-00126-000",
    "BraTS-GLI-00127-000",
    "BraTS-GLI-00131-000",
    "BraTS-GLI-00132-000",
    "BraTS-GLI-00133-000",
    "BraTS-GLI-00136-000",
    "BraTS-GLI-00137-000",
    "BraTS-GLI-00144-000",
    "BraTS-GLI-00149-000",
    "BraTS-GLI-00152-000",
    "BraTS-GLI-00154-000",
    "BraTS-GLI-00157-000",
    "BraTS-GLI-00158-000",
    "BraTS-GLI-00159-000",
    "BraTS-GLI-00160-000",
    "BraTS-GLI-00166-000",
    "BraTS-GLI-00167-000",
    "BraTS-GLI-00170-000",
    "BraTS-GLI-00172-000",
    "BraTS-GLI-00176-000",
    "BraTS-GLI-00177-000",
    "BraTS-GLI-00185-000",
    "BraTS-GLI-00186-000",
    "BraTS-GLI-00191-000",
    "BraTS-GLI-00201-000",
    "BraTS-GLI-00203-000",
    "BraTS-GLI-00204-000",
    "BraTS-GLI-00207-000",
    "BraTS-GLI-00209-000",
    "BraTS-GLI-00214-000",
    "BraTS-GLI-00216-000",
    "BraTS-GLI-00217-000",
    "BraTS-GLI-00227-000",
    "BraTS-GLI-00228-000",
    "BraTS-GLI-00233-000",
    "BraTS-GLI-00237-000",
    "BraTS-GLI-00239-000",
    "BraTS-GLI-00240-000",
    "BraTS-GLI-00243-000",
    "BraTS-GLI-00249-000",
    "BraTS-GLI-00250-000",
    "BraTS-GLI-00253-000",
    "BraTS-GLI-00254-000",
    "BraTS-GLI-00258-000",
    "BraTS-GLI-00260-000",
    "BraTS-GLI-00271-000",
    "BraTS-GLI-00274-000",
    "BraTS-GLI-00280-000",
    "BraTS-GLI-00281-000",
    "BraTS-GLI-00282-000",
    "BraTS-GLI-00285-000",
    "BraTS-GLI-00288-000",
    "BraTS-GLI-00290-000",
    "BraTS-GLI-00292-000",
    "BraTS-GLI-00293-000",
    "BraTS-GLI-00298-000",
    "BraTS-GLI-00299-000",
    "BraTS-GLI-00301-000",
    "BraTS-GLI-00304-000",
    "BraTS-GLI-00313-000",
    "BraTS-GLI-00318-000",
    "BraTS-GLI-00321-000",
    "BraTS-GLI-00322-000",
    "BraTS-GLI-00325-000",
    "BraTS-GLI-00329-000",
    "BraTS-GLI-00331-000",
    "BraTS-GLI-00336-000",
    "BraTS-GLI-00338-000",
    "BraTS-GLI-00346-000",
    "BraTS-GLI-00349-000",
    "BraTS-GLI-00350-000",
    "BraTS-GLI-00366-000",
    "BraTS-GLI-00367-000",
    "BraTS-GLI-00371-000",
    "BraTS-GLI-00377-000",
    "BraTS-GLI-00378-000",
    "BraTS-GLI-00379-000",
    "BraTS-GLI-00380-000",
    "BraTS-GLI-00390-000",
    "BraTS-GLI-00395-000",
    "BraTS-GLI-00399-000",
    "BraTS-GLI-00400-000",
    "BraTS-GLI-00401-000",
    "BraTS-GLI-00402-000",
    "BraTS-GLI-00405-000",
    "BraTS-GLI-00406-000",
    "BraTS-GLI-00413-000",
    "BraTS-GLI-00416-000",
    "BraTS-GLI-00418-000",
    "BraTS-GLI-00425-000",
    "BraTS-GLI-00429-000",
    "BraTS-GLI-00441-000",
    "BraTS-GLI-00442-000",
    "BraTS-GLI-00443-000",
    "BraTS-GLI-00446-000",
    "BraTS-GLI-00449-000",
    "BraTS-GLI-00455-000",
    "BraTS-GLI-00457-000",
    "BraTS-GLI-00459-000",
    "BraTS-GLI-00464-000",
    "BraTS-GLI-00464-001",
    "BraTS-GLI-00469-001",
    "BraTS-GLI-00480-000",
    "BraTS-GLI-00483-000",
    "BraTS-GLI-00483-001",
    "BraTS-GLI-00485-000",
    "BraTS-GLI-00485-001",
    "BraTS-GLI-00491-000",
    "BraTS-GLI-00491-001",
    "BraTS-GLI-00495-000",
    "BraTS-GLI-00502-000",
    "BraTS-GLI-00504-000",
    "BraTS-GLI-00507-000",
    "BraTS-GLI-00512-000",
    "BraTS-GLI-00514-000",
    "BraTS-GLI-00514-001",
    "BraTS-GLI-00517-001",
    "BraTS-GLI-00524-000",
    "BraTS-GLI-00525-000",
    "BraTS-GLI-00528-000",
    "BraTS-GLI-00529-000",
    "BraTS-GLI-00530-000",
    "BraTS-GLI-00533-000",
    "BraTS-GLI-00537-000",
    "BraTS-GLI-00538-000",
    "BraTS-GLI-00539-000",
    "BraTS-GLI-00544-001",
    "BraTS-GLI-00550-001",
    "BraTS-GLI-00552-000",
    "BraTS-GLI-00554-000",
    "BraTS-GLI-00558-000",
    "BraTS-GLI-00559-000",
    "BraTS-GLI-00561-000",
    "BraTS-GLI-00567-000",
    "BraTS-GLI-00572-000",
    "BraTS-GLI-00574-000",
    "BraTS-GLI-00578-000",
    "BraTS-GLI-00580-000",
    "BraTS-GLI-00584-000",
    "BraTS-GLI-00587-000",
    "BraTS-GLI-00588-000",
    "BraTS-GLI-00589-000",
    "BraTS-GLI-00591-000",
    "BraTS-GLI-00596-000",
    "BraTS-GLI-00602-000",
    "BraTS-GLI-00602-001",
    "BraTS-GLI-00608-001",
    "BraTS-GLI-00616-000",
    "BraTS-GLI-00618-000",
    "BraTS-GLI-00619-000",
    "BraTS-GLI-00620-000",
    "BraTS-GLI-00620-001",
    "BraTS-GLI-00623-000",
    "BraTS-GLI-00625-000",
    "BraTS-GLI-00626-000",
    "BraTS-GLI-00636-000",
    "BraTS-GLI-00636-001",
    "BraTS-GLI-00640-000",
    "BraTS-GLI-00641-000",
    "BraTS-GLI-00641-001",
    "BraTS-GLI-00649-000",
    "BraTS-GLI-00650-000",
    "BraTS-GLI-00651-000",
    "BraTS-GLI-00654-000",
    "BraTS-GLI-00654-001",
    "BraTS-GLI-00655-000",
    "BraTS-GLI-00661-000",
    "BraTS-GLI-00668-000",
    "BraTS-GLI-00674-000",
    "BraTS-GLI-00675-000",
    "BraTS-GLI-00676-001",
    "BraTS-GLI-00677-000",
    "BraTS-GLI-00679-000",
    "BraTS-GLI-00684-000",
    "BraTS-GLI-00687-000",
    "BraTS-GLI-00688-000",
    "BraTS-GLI-00689-000",
    "BraTS-GLI-00689-001",
    "BraTS-GLI-00692-000",
    "BraTS-GLI-00692-001",
    "BraTS-GLI-00693-000",
    "BraTS-GLI-00697-000",
    "BraTS-GLI-00706-000",
    "BraTS-GLI-00714-001",
    "BraTS-GLI-00716-000",
    "BraTS-GLI-00727-001",
    "BraTS-GLI-00729-000",
    "BraTS-GLI-00729-001",
    "BraTS-GLI-00730-000",
    "BraTS-GLI-00730-001",
    "BraTS-GLI-00731-001",
    "BraTS-GLI-00732-000",
    "BraTS-GLI-00733-001",
    "BraTS-GLI-00734-001",
    "BraTS-GLI-00735-001",
    "BraTS-GLI-00736-001",
    "BraTS-GLI-00739-000",
    "BraTS-GLI-00744-000",
    "BraTS-GLI-00750-000",
    "BraTS-GLI-00753-000",
    "BraTS-GLI-00756-001",
    "BraTS-GLI-00760-000",
    "BraTS-GLI-00772-000",
    "BraTS-GLI-00772-001",
    "BraTS-GLI-00773-000",
    "BraTS-GLI-00777-000",
    "BraTS-GLI-00787-000",
    "BraTS-GLI-00792-000",
    "BraTS-GLI-00799-000",
    "BraTS-GLI-00800-000",
    "BraTS-GLI-00804-000",
    "BraTS-GLI-00807-000",
    "BraTS-GLI-00808-000",
    "BraTS-GLI-00814-000",
    "BraTS-GLI-00823-000",
    "BraTS-GLI-00824-000",
    "BraTS-GLI-00828-000",
    "BraTS-GLI-00830-000",
    "BraTS-GLI-00837-000",
    "BraTS-GLI-00838-000",
    "BraTS-GLI-00839-000",
    "BraTS-GLI-01001-000",
    "BraTS-GLI-01009-000",
    "BraTS-GLI-01011-000",
    "BraTS-GLI-01014-000",
    "BraTS-GLI-01015-000",
    "BraTS-GLI-01017-000",
    "BraTS-GLI-01020-000",
    "BraTS-GLI-01022-000",
    "BraTS-GLI-01024-000",
    "BraTS-GLI-01027-001",
    "BraTS-GLI-01033-000",
    "BraTS-GLI-01035-000",
    "BraTS-GLI-01036-000",
    "BraTS-GLI-01037-000",
    "BraTS-GLI-01039-000",
    "BraTS-GLI-01044-000",
    "BraTS-GLI-01054-000",
    "BraTS-GLI-01055-000",
    "BraTS-GLI-01057-000",
    "BraTS-GLI-01058-000",
    "BraTS-GLI-01061-000",
    "BraTS-GLI-01062-000",
    "BraTS-GLI-01063-000",
    "BraTS-GLI-01064-000",
    "BraTS-GLI-01066-000",
    "BraTS-GLI-01071-000",
    "BraTS-GLI-01074-000",
    "BraTS-GLI-01075-000",
    "BraTS-GLI-01079-000",
    "BraTS-GLI-01083-000",
    "BraTS-GLI-01088-000",
    "BraTS-GLI-01089-000",
    "BraTS-GLI-01092-000",
    "BraTS-GLI-01094-000",
    "BraTS-GLI-01096-000",
    "BraTS-GLI-01097-000",
    "BraTS-GLI-01099-000",
    "BraTS-GLI-01100-000",
    "BraTS-GLI-01103-000",
    "BraTS-GLI-01104-000",
    "BraTS-GLI-01106-000",
    "BraTS-GLI-01107-000",
    "BraTS-GLI-01113-000",
    "BraTS-GLI-01115-000",
    "BraTS-GLI-01116-000",
    "BraTS-GLI-01118-000",
    "BraTS-GLI-01121-000",
    "BraTS-GLI-01122-000",
    "BraTS-GLI-01123-000",
    "BraTS-GLI-01125-000",
    "BraTS-GLI-01126-000",
    "BraTS-GLI-01128-000",
    "BraTS-GLI-01129-000",
    "BraTS-GLI-01131-000",
    "BraTS-GLI-01132-000",
    "BraTS-GLI-01137-000",
    "BraTS-GLI-01141-000",
    "BraTS-GLI-01145-000",
    "BraTS-GLI-01151-000",
    "BraTS-GLI-01153-000",
    "BraTS-GLI-01157-000",
    "BraTS-GLI-01160-000",
    "BraTS-GLI-01163-000",
    "BraTS-GLI-01164-000",
    "BraTS-GLI-01168-000",
    "BraTS-GLI-01171-000",
    "BraTS-GLI-01173-000",
    "BraTS-GLI-01178-000",
    "BraTS-GLI-01179-000",
    "BraTS-GLI-01181-000",
    "BraTS-GLI-01182-000",
    "BraTS-GLI-01185-000",
    "BraTS-GLI-01186-000",
    "BraTS-GLI-01194-000",
    "BraTS-GLI-01195-000",
    "BraTS-GLI-01197-000",
    "BraTS-GLI-01198-000",
    "BraTS-GLI-01203-000",
    "BraTS-GLI-01205-000",
    "BraTS-GLI-01207-000",
    "BraTS-GLI-01209-000",
    "BraTS-GLI-01210-000",
    "BraTS-GLI-01213-000",
    "BraTS-GLI-01221-000",
    "BraTS-GLI-01223-000",
    "BraTS-GLI-01224-000",
    "BraTS-GLI-01229-000",
    "BraTS-GLI-01231-000",
    "BraTS-GLI-01235-000",
    "BraTS-GLI-01236-000",
    "BraTS-GLI-01238-000",
    "BraTS-GLI-01239-000",
    "BraTS-GLI-01242-000",
    "BraTS-GLI-01243-000",
    "BraTS-GLI-01247-000",
    "BraTS-GLI-01254-000",
    "BraTS-GLI-01255-000",
    "BraTS-GLI-01256-000",
    "BraTS-GLI-01257-000",
    "BraTS-GLI-01258-000",
    "BraTS-GLI-01260-000",
    "BraTS-GLI-01261-000",
    "BraTS-GLI-01263-000",
    "BraTS-GLI-01264-000",
    "BraTS-GLI-01270-000",
    "BraTS-GLI-01272-000",
    "BraTS-GLI-01273-000",
    "BraTS-GLI-01275-000",
    "BraTS-GLI-01276-000",
    "BraTS-GLI-01278-000",
    "BraTS-GLI-01283-000",
    "BraTS-GLI-01284-000",
    "BraTS-GLI-01285-000",
    "BraTS-GLI-01290-000",
    "BraTS-GLI-01291-000",
    "BraTS-GLI-01292-000",
    "BraTS-GLI-01294-000",
    "BraTS-GLI-01295-000",
    "BraTS-GLI-01297-000",
    "BraTS-GLI-01299-000",
    "BraTS-GLI-01300-000",
    "BraTS-GLI-01307-000",
    "BraTS-GLI-01310-000",
    "BraTS-GLI-01316-000",
    "BraTS-GLI-01317-000",
    "BraTS-GLI-01323-000",
    "BraTS-GLI-01324-000",
    "BraTS-GLI-01326-000",
    "BraTS-GLI-01330-000",
    "BraTS-GLI-01332-000",
    "BraTS-GLI-01337-000",
    "BraTS-GLI-01339-000",
    "BraTS-GLI-01345-000",
    "BraTS-GLI-01347-000",
    "BraTS-GLI-01348-000",
    "BraTS-GLI-01349-000",
    "BraTS-GLI-01350-000",
    "BraTS-GLI-01354-000",
    "BraTS-GLI-01357-000",
    "BraTS-GLI-01364-000",
    "BraTS-GLI-01365-000",
    "BraTS-GLI-01367-000",
    "BraTS-GLI-01370-000",
    "BraTS-GLI-01371-000",
    "BraTS-GLI-01372-000",
    "BraTS-GLI-01374-000",
    "BraTS-GLI-01375-000",
    "BraTS-GLI-01376-000",
    "BraTS-GLI-01381-000",
    "BraTS-GLI-01383-000",
    "BraTS-GLI-01387-000",
    "BraTS-GLI-01388-000",
    "BraTS-GLI-01391-000",
    "BraTS-GLI-01393-000",
    "BraTS-GLI-01395-000",
    "BraTS-GLI-01397-000",
    "BraTS-GLI-01398-000",
    "BraTS-GLI-01399-000",
    "BraTS-GLI-01400-000",
    "BraTS-GLI-01402-000",
    "BraTS-GLI-01403-000",
    "BraTS-GLI-01405-000",
    "BraTS-GLI-01406-000",
    "BraTS-GLI-01411-000",
    "BraTS-GLI-01415-000",
    "BraTS-GLI-01416-000",
    "BraTS-GLI-01419-000",
    "BraTS-GLI-01422-000",
    "BraTS-GLI-01423-000",
    "BraTS-GLI-01427-000",
    "BraTS-GLI-01431-000",
    "BraTS-GLI-01432-000",
    "BraTS-GLI-01433-000",
    "BraTS-GLI-01436-000",
    "BraTS-GLI-01437-000",
    "BraTS-GLI-01439-000",
    "BraTS-GLI-01440-000",
    "BraTS-GLI-01443-000",
    "BraTS-GLI-01444-000",
    "BraTS-GLI-01450-000",
    "BraTS-GLI-01453-000",
    "BraTS-GLI-01459-000",
    "BraTS-GLI-01460-000",
    "BraTS-GLI-01462-000",
    "BraTS-GLI-01463-000",
    "BraTS-GLI-01466-000",
    "BraTS-GLI-01468-000",
    "BraTS-GLI-01470-000",
    "BraTS-GLI-01474-000",
    "BraTS-GLI-01475-000",
    "BraTS-GLI-01481-000",
    "BraTS-GLI-01482-000",
    "BraTS-GLI-01485-000",
    "BraTS-GLI-01486-000",
    "BraTS-GLI-01488-000",
    "BraTS-GLI-01490-000",
    "BraTS-GLI-01494-000",
    "BraTS-GLI-01496-000",
    "BraTS-GLI-01498-000",
    "BraTS-GLI-01499-000",
    "BraTS-GLI-01501-000",
    "BraTS-GLI-01502-000",
    "BraTS-GLI-01504-000",
    "BraTS-GLI-01505-000",
    "BraTS-GLI-01507-000",
    "BraTS-GLI-01508-000",
    "BraTS-GLI-01509-000",
    "BraTS-GLI-01510-000",
    "BraTS-GLI-01511-000",
    "BraTS-GLI-01512-000",
    "BraTS-GLI-01515-000",
    "BraTS-GLI-01516-000",
    "BraTS-GLI-01517-000",
    "BraTS-GLI-01519-000",
    "BraTS-GLI-01520-000",
    "BraTS-GLI-01523-000",
    "BraTS-GLI-01524-000",
    "BraTS-GLI-01526-000",
    "BraTS-GLI-01529-000",
    "BraTS-GLI-01530-000",
    "BraTS-GLI-01533-000",
    "BraTS-GLI-01536-000",
    "BraTS-GLI-01537-000",
    "BraTS-GLI-01610-000",
    "BraTS-GLI-01657-000",
    "BraTS-GLI-01660-000",
    "BraTS-GLI-01662-000",
    "BraTS-GLI-01664-000",
    "BraTS-GLI-01665-000",
]

VAL = [
    "BraTS-GLI-00005-000",
    "BraTS-GLI-00035-000",
    "BraTS-GLI-00084-000",
    "BraTS-GLI-00090-000",
    "BraTS-GLI-00097-000",
    "BraTS-GLI-00101-000",
    "BraTS-GLI-00140-000",
    "BraTS-GLI-00147-000",
    "BraTS-GLI-00199-000",
    "BraTS-GLI-00251-000",
    "BraTS-GLI-00259-000",
    "BraTS-GLI-00273-000",
    "BraTS-GLI-00283-000",
    "BraTS-GLI-00284-000",
    "BraTS-GLI-00296-000",
    "BraTS-GLI-00297-000",
    "BraTS-GLI-00300-000",
    "BraTS-GLI-00370-000",
    "BraTS-GLI-00388-000",
    "BraTS-GLI-00389-000",
    "BraTS-GLI-00391-000",
    "BraTS-GLI-00419-000",
    "BraTS-GLI-00421-000",
    "BraTS-GLI-00440-000",
    "BraTS-GLI-00451-000",
    "BraTS-GLI-00468-000",
    "BraTS-GLI-00479-000",
    "BraTS-GLI-00496-000",
    "BraTS-GLI-00499-001",
    "BraTS-GLI-00510-001",
    "BraTS-GLI-00511-001",
    "BraTS-GLI-00516-000",
    "BraTS-GLI-00518-000",
    "BraTS-GLI-00542-000",
    "BraTS-GLI-00545-000",
    "BraTS-GLI-00555-000",
    "BraTS-GLI-00555-001",
    "BraTS-GLI-00557-000",
    "BraTS-GLI-00563-000",
    "BraTS-GLI-00576-000",
    "BraTS-GLI-00583-000",
    "BraTS-GLI-00594-000",
    "BraTS-GLI-00605-000",
    "BraTS-GLI-00606-000",
    "BraTS-GLI-00611-000",
    "BraTS-GLI-00611-001",
    "BraTS-GLI-00613-001",
    "BraTS-GLI-00615-000",
    "BraTS-GLI-00652-000",
    "BraTS-GLI-00680-001",
    "BraTS-GLI-00683-000",
    "BraTS-GLI-00683-001",
    "BraTS-GLI-00686-000",
    "BraTS-GLI-00707-000",
    "BraTS-GLI-00715-000",
    "BraTS-GLI-00725-001",
    "BraTS-GLI-00740-000",
    "BraTS-GLI-00751-000",
    "BraTS-GLI-00751-001",
    "BraTS-GLI-00764-000",
    "BraTS-GLI-00784-000",
    "BraTS-GLI-01003-000",
    "BraTS-GLI-01034-000",
    "BraTS-GLI-01110-000",
    "BraTS-GLI-01111-000",
    "BraTS-GLI-01112-000",
    "BraTS-GLI-01142-000",
    "BraTS-GLI-01143-000",
    "BraTS-GLI-01161-000",
    "BraTS-GLI-01174-000",
    "BraTS-GLI-01183-000",
    "BraTS-GLI-01184-000",
    "BraTS-GLI-01187-000",
    "BraTS-GLI-01190-000",
    "BraTS-GLI-01193-000",
    "BraTS-GLI-01199-000",
    "BraTS-GLI-01201-000",
    "BraTS-GLI-01202-000",
    "BraTS-GLI-01204-000",
    "BraTS-GLI-01206-000",
    "BraTS-GLI-01217-000",
    "BraTS-GLI-01225-000",
    "BraTS-GLI-01241-000",
    "BraTS-GLI-01246-000",
    "BraTS-GLI-01248-000",
    "BraTS-GLI-01280-000",
    "BraTS-GLI-01282-000",
    "BraTS-GLI-01288-000",
    "BraTS-GLI-01314-000",
    "BraTS-GLI-01336-000",
    "BraTS-GLI-01344-000",
    "BraTS-GLI-01401-000",
    "BraTS-GLI-01412-000",
    "BraTS-GLI-01424-000",
    "BraTS-GLI-01434-000",
    "BraTS-GLI-01458-000",
    "BraTS-GLI-01465-000",
    "BraTS-GLI-01467-000",
    "BraTS-GLI-01480-000",
    "BraTS-GLI-01658-000",
]

SUBSET = TRAIN + VAL
print("subset cases: train", len(TRAIN), "val", len(VAL), "total", len(SUBSET))

In [ ]:
# Step 3 — build a symlink raw folder for ONLY the 600 subset cases
import os, glob
# Kaggle's mount path varies, so locate the dataset by the folder that holds
# dataset.json + imagesTr.
MOUNT = None
for cand in glob.glob("/kaggle/input/**/dataset.json", recursive=True):
    d = os.path.dirname(cand)
    if os.path.isdir(os.path.join(d, "imagesTr")):
        MOUNT = d; break
assert MOUNT, "could not locate dataset mount under /kaggle/input"
print("MOUNT =", MOUNT)

RAW = "/kaggle/working/nnUNet_raw/Dataset501_BraTS2023GLI"
for f in ["imagesTr", "labelsTr"]:
    os.makedirs(os.path.join(RAW, f), exist_ok=True)

def link_one(src, link):
    # flat file -> link directly; nested dir (X.nii/inner.nii) -> link the single inner file
    if os.path.islink(link) or os.path.exists(link):
        os.remove(link)
    if os.path.isfile(src):
        os.symlink(src, link); return "file"
    if os.path.isdir(src):
        inner = os.listdir(src)
        assert len(inner) == 1, f"{src} contains {len(inner)} files: {inner[:3]}"
        os.symlink(os.path.join(src, inner[0]), link); return "dir"
    raise FileNotFoundError(src)

n_img = n_lbl = 0
kinds = set()
for cid in SUBSET:
    for ch in ("0000", "0001", "0002", "0003"):
        name = f"{cid}_{ch}.nii"
        kinds.add(link_one(os.path.join(MOUNT, "imagesTr", name),
                           os.path.join(RAW, "imagesTr", name)))
        n_img += 1
    lname = f"{cid}.nii"
    link_one(os.path.join(MOUNT, "labelsTr", lname), os.path.join(RAW, "labelsTr", lname))
    n_lbl += 1
print(f"linked images={n_img} (expect 2400), labels={n_lbl} (expect 600), image kinds={kinds}")
assert n_img == 2400 and n_lbl == 600

In [ ]:
# Step 4 — channel-name check on the subset's nested GLI images
# _0000 -> t1 (NOT t1ce/t1c), _0001 -> t1ce/t1c, _0002 -> t2, _0003 -> flair
from collections import Counter
EXPECT = {"0000": {"t1"}, "0001": {"t1ce","t1c"}, "0002": {"t2"}, "0003": {"flair"}}
def modality_of(inner): return inner.rsplit("_",1)[-1].replace(".nii","").lower()

counts, mismatches = Counter(), []
for cid in SUBSET:
    for ch in ("0000","0001","0002","0003"):
        entry = f"{cid}_{ch}.nii"
        src = os.path.join(MOUNT, "imagesTr", entry)
        if not os.path.isdir(src):     # flat image carries no modality token -> skip
            continue
        mod = modality_of(os.listdir(src)[0])
        ok  = mod in EXPECT[ch]
        counts[(ch, mod, ok)] += 1
        if not ok:
            mismatches.append((entry, ch, mod))

print("channel / modality / ok  ->  count")
for k in sorted(counts):
    print(f"  {k[0]}  {k[1]:6s}  {str(k[2]):5s}  -> {counts[k]}")
if mismatches:
    for m in mismatches[:20]:
        print("MISMATCH:", m)
    raise RuntimeError(f"Channel check FAILED: {len(mismatches)} mismatched images")
print("Channel check PASSED for all subset GLI images.")

In [ ]:
# Step 5 — write dataset.json with file_ending = ".nii" and numTraining = 600
import json
ds = json.load(open(os.path.join(MOUNT, "dataset.json")))
ds["file_ending"]  = ".nii"
ds["numTraining"]  = 600
json.dump(ds, open(os.path.join(RAW, "dataset.json"), "w"), indent=2)
print(json.dumps(ds, indent=2))

In [ ]:
# Step 6 — verify counts + load 3 random subset training cases
import nibabel as nib, numpy as np, random
for f, exp in [("imagesTr", 2400), ("labelsTr", 600)]:
    got = len(os.listdir(os.path.join(RAW, f)))
    print(f"{f:10s} {got:5d} (expect {exp}) [{'OK' if got==exp else 'MISMATCH'}]")
    assert got == exp

random.seed(0)
for cid in random.sample(SUBSET, 3):
    shapes = [nib.load(os.path.join(RAW,"imagesTr",f"{cid}_{i:04d}.nii")).shape for i in range(4)]
    seg = nib.load(os.path.join(RAW,"labelsTr",f"{cid}.nii")).get_fdata().astype(int)
    print(cid, "4-channel shapes:", shapes, "| label values:", sorted(np.unique(seg).tolist()))

In [ ]:
# Step 7 — nnU-Net environment variables
os.environ["nnUNet_raw"]          = "/kaggle/working/nnUNet_raw"
os.environ["nnUNet_preprocessed"] = "/kaggle/working/nnUNet_preprocessed"
os.environ["nnUNet_results"]      = "/kaggle/working/nnUNet_results"
for k in ["nnUNet_raw","nnUNet_preprocessed","nnUNet_results"]:
    os.makedirs(os.environ[k], exist_ok=True)
    print(k, "=", os.environ[k])

In [ ]:
# Step 8 — plan & preprocess (3d_fullres), timed
import time
cmd = ["nnUNetv2_plan_and_preprocess","-d","501","-c","3d_fullres",
       "--verify_dataset_integrity","-np","4"]
print("running:", " ".join(cmd), flush=True)
t0 = time.time()
p = subprocess.run(cmd, env=os.environ, capture_output=True, text=True)
dt = time.time() - t0
print(f"RETURN CODE: {p.returncode}")
print(f"ELAPSED: {dt/60:.1f} min ({dt:.0f} s)")
print("----- STDOUT (tail 80) -----")
print("\n".join(p.stdout.splitlines()[-80:]))
print("----- STDERR (tail 40) -----")
print("\n".join(p.stderr.splitlines()[-40:]))
assert p.returncode == 0, "nnUNetv2_plan_and_preprocess FAILED (see stderr above)"
PP = os.path.join(os.environ["nnUNet_preprocessed"], "Dataset501_BraTS2023GLI")

In [ ]:
# Step 9 — write splits_final.json (the exact 600-case one fold) into preprocessed
import json
os.makedirs(PP, exist_ok=True)
json.dump([{"train": TRAIN, "val": VAL}], open(os.path.join(PP, "splits_final.json"), "w"), indent=2)
print("wrote splits_final.json (train", len(TRAIN), "val", len(VAL), ") ->", PP)
print(sorted(os.listdir(PP)))

In [ ]:
# Step 10 — print the 3d_fullres plan + preprocessed size; FAIL LOUDLY if > 17 GB
import json
plans = json.load(open(os.path.join(PP, "nnUNetPlans.json")))
cfg   = plans["configurations"]["3d_fullres"]
arch  = cfg.get("architecture", {}); ak = arch.get("arch_kwargs", {})
print("channel_names      :", json.load(open(os.path.join(RAW,"dataset.json")))["channel_names"])
print("patch_size         :", cfg.get("patch_size"))
print("batch_size         :", cfg.get("batch_size"))
print("spacing            :", cfg.get("spacing"))
print("normalization      :", cfg.get("normalization_schemes"))
print("n_stages           :", ak.get("n_stages"))
print("features_per_stage :", ak.get("features_per_stage"))
print("strides            :", ak.get("strides"))
print("network_class      :", arch.get("network_class_name"))

nb_bytes = int(subprocess.run(["du","-sb", os.environ["nnUNet_preprocessed"]],
                              capture_output=True, text=True).stdout.split()[0])
gb = nb_bytes / 1e9
print(subprocess.run(["du","-sh", os.environ["nnUNet_preprocessed"]], capture_output=True, text=True).stdout.strip())
print(f"nnUNet_preprocessed = {gb:.2f} GB")
assert gb <= 17, f"preprocessed size {gb:.2f} GB exceeds the 17 GB cap"
print("SIZE OK (<= 17 GB)")